In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv("customer_churn_clean.csv")

In [3]:
df.head()

,CustomerID,Count,Country,State,City,Zip_Code,Lat_Long,Latitude,Longitude,Gender,Senior_Citizen,Partner,Dependents,Tenure_Months,Phone_Service,Multiple_Lines,Internet_Service,Online_Security,Online_Backup,Device_Protection,Tech_Support,Streaming_TV,Streaming_Movies,Contract,Paperless_Billing,Payment_Method,Monthly_Charges,Total_Charges,Churn_Label,Churn_Value,Churn_Score,CLTV,Churn_Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [4]:
df.shape

(7043, 33)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip_Code           7043 non-null   int64  
 6   Lat_Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior_Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure_Months      7043 non-null   int64  
 14  Phone_Service      7043 non-null   object 
 15  Multiple_Lines     7043 non-null   object 
 16  Internet_Service   7043 

In [6]:
df["Tenure_Group"] = pd.cut(
    df["Tenure_Months"],
    bins=[-1,12,24,48,72],
    labels=["0-12 Months",
            "13-24 Months",
            "25-48 Months",
            "49-72 Months"]
)

In [7]:
df["Tenure_Group"].value_counts()

Tenure_Group
49-72 Months    2239
0-12 Months     2186
25-48 Months    1594
13-24 Months    1024
Name: count, dtype: int64

In [8]:
df["Monthly_Charge_Category"] = pd.cut(
    df["Monthly_Charges"],
    bins=[0,35,70,120],
    labels=["Low","Medium","High"]
)

In [9]:
df["Monthly_Charge_Category"].value_counts()

Monthly_Charge_Category
High      3583
Low       1735
Medium    1725
Name: count, dtype: int64

In [10]:
df["CLTV_Category"] = pd.qcut(
    df["CLTV"],
    q=3,
    labels=["Low","Medium","High"]
)

In [11]:
df["CLTV_Category"].value_counts()

CLTV_Category
Medium    2349
Low       2348
High      2346
Name: count, dtype: int64

In [12]:
service_columns = [

"Phone_Service",

"Multiple_Lines",

"Online_Security",

"Online_Backup",

"Device_Protection",

"Tech_Support",

"Streaming_TV",

"Streaming_Movies"

]

In [13]:
df["Total_Services"] = (
    df[service_columns] == "Yes"
).sum(axis=1)

In [14]:
df["Total_Services"].describe()

count    7043.000000
mean        3.362914
std         2.062031
min         0.000000
25%         1.000000
50%         3.000000
75%         5.000000
max         8.000000
Name: Total_Services, dtype: float64

In [15]:
average_cltv = df["CLTV"].mean()

df["High_Value_Customer"] = np.where(
    df["CLTV"] >= average_cltv,
    "Yes",
    "No"
)

In [16]:
df["High_Value_Customer"].value_counts()

High_Value_Customer
Yes    3817
No     3226
Name: count, dtype: int64

In [17]:
def customer_segment(cltv):

    if cltv >= 5000:
        return "Premium"

    elif cltv >= 4000:
        return "Gold"

    elif cltv >= 3000:
        return "Silver"

    else:
        return "Standard"

df["Customer_Segment"] = df["CLTV"].apply(customer_segment)

In [18]:
df["Customer_Segment"].value_counts()

Customer_Segment
Premium     2574
Gold        2102
Silver      1188
Standard    1179
Name: count, dtype: int64

In [19]:
def risk_level(score):

    if score >= 80:
        return "High Risk"

    elif score >= 50:
        return "Medium Risk"

    else:
        return "Low Risk"

df["Risk_Level"] = df["Churn_Score"].apply(risk_level)

In [20]:
df["Risk_Level"].value_counts()

Risk_Level
Medium Risk    3309
Low Risk       2533
High Risk      1201
Name: count, dtype: int64

In [23]:
df["Total_Charges"] = pd.to_numeric(df["Total_Charges"], errors="coerce")

In [24]:
df["Total_Charges"] = df["Total_Charges"].fillna(0)

In [25]:
df["Total_Charges"].dtype

dtype('float64')

In [26]:
df["Revenue_Category"] = pd.qcut(
    df["Total_Charges"],
    q=3,
    labels=["Low", "Medium", "High"]
)

In [27]:
df["Revenue_Category"].value_counts()

Revenue_Category
Low       2348
High      2348
Medium    2347
Name: count, dtype: int64

In [28]:
df["Customer_Age_Group"] = np.where(
    df["Senior_Citizen"]==1,
    "Senior Citizen",
    "Adult"
)

In [29]:
df["Customer_Age_Group"].value_counts()

Customer_Age_Group
Adult    7043
Name: count, dtype: int64

In [30]:
df["Customer_Status"] = np.where(
    df["Churn_Label"]=="Yes",
    "Inactive",
    "Active"
)

In [31]:
df["Customer_Status"].value_counts()

Customer_Status
Active      5174
Inactive    1869
Name: count, dtype: int64

In [32]:
df.head()

,CustomerID,Count,Country,State,City,Zip_Code,Lat_Long,Latitude,Longitude,Gender,Senior_Citizen,Partner,Dependents,Tenure_Months,Phone_Service,Multiple_Lines,Internet_Service,Online_Security,Online_Backup,Device_Protection,Tech_Support,Streaming_TV,Streaming_Movies,Contract,Paperless_Billing,Payment_Method,Monthly_Charges,Total_Charges,Churn_Label,Churn_Value,Churn_Score,CLTV,Churn_Reason,Tenure_Group,Monthly_Charge_Category,CLTV_Category,Total_Services,High_Value_Customer,Customer_Segment,Risk_Level,Revenue_Category,Customer_Age_Group,Customer_Status
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer,0-12 Months,Medium,Low,3,No,Silver,High Risk,Low,Adult,Inactive
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved,0-12 Months,High,Low,1,No,Standard,Medium Risk,Low,Adult,Inactive
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,1,86,5372,Moved,0-12 Months,High,High,5,Yes,Premium,High Risk,Medium,Adult,Inactive
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved,25-48 Months,High,Medium,6,Yes,Premium,High Risk,High,Adult,Inactive
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,1,89,5340,Competitor had better devices,49-72 Months,High,High,6,Yes,Premium,High Risk,High,Adult,Inactive


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 43 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   CustomerID               7043 non-null   object  
 1   Count                    7043 non-null   int64   
 2   Country                  7043 non-null   object  
 3   State                    7043 non-null   object  
 4   City                     7043 non-null   object  
 5   Zip_Code                 7043 non-null   int64   
 6   Lat_Long                 7043 non-null   object  
 7   Latitude                 7043 non-null   float64 
 8   Longitude                7043 non-null   float64 
 9   Gender                   7043 non-null   object  
 10  Senior_Citizen           7043 non-null   object  
 11  Partner                  7043 non-null   object  
 12  Dependents               7043 non-null   object  
 13  Tenure_Months            7043 non-null   int64   
 14  Phone_Se

In [34]:
df.columns

Index(['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip_Code',
       'Lat_Long', 'Latitude', 'Longitude', 'Gender', 'Senior_Citizen',
       'Partner', 'Dependents', 'Tenure_Months', 'Phone_Service',
       'Multiple_Lines', 'Internet_Service', 'Online_Security',
       'Online_Backup', 'Device_Protection', 'Tech_Support', 'Streaming_TV',
       'Streaming_Movies', 'Contract', 'Paperless_Billing', 'Payment_Method',
       'Monthly_Charges', 'Total_Charges', 'Churn_Label', 'Churn_Value',
       'Churn_Score', 'CLTV', 'Churn_Reason', 'Tenure_Group',
       'Monthly_Charge_Category', 'CLTV_Category', 'Total_Services',
       'High_Value_Customer', 'Customer_Segment', 'Risk_Level',
       'Revenue_Category', 'Customer_Age_Group', 'Customer_Status'],
      dtype='object')

In [35]:
df.to_csv(
    "downloads/customer_churn_feature_engineered.csv",
    index=False
)